# 07 Model Comparison

Multi-model benchmark prioritizing recall and ROC-AUC for high-cost false negative attrition.

### 1. Comparative Evaluation across LR, Random Forest, and XGBoost

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv("../data/processed/employee_attrition_processed.csv")
df['IncomePerYearAtCompany'] = df['MonthlyIncome'] / (df['YearsAtCompany'] + 1.0)
df['PromotionLagRatio'] = df['YearsSinceLastPromotion'] / (df['YearsInCurrentRole'] + 1.0)
df['TotalSatisfactionScore'] = df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['RelationshipSatisfaction'] + df['WorkLifeBalance']
df['ExperienceRatio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1.0)

X = df.drop(columns=['EmployeeID', 'Attrition'])
y = (df['Attrition'] == 'Yes').astype(int)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=8, class_weight='balanced', random_state=42),
    'XGBoost': XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.08, scale_pos_weight=3.5, random_state=42, eval_metric='logloss')
}

res_list = []
for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    
    res_list.append({
        'Model': name,
        'Precision': precision_score(y_test, preds, zero_division=0),
        'Recall': recall_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs)
    })

comp_df = pd.DataFrame(res_list)
print("=== Model Comparison Table ===")
print(comp_df.to_string(index=False))